# PatchBench FR/EN sur Kaggle (2 GPU T4 en parallèle)

**Avant de commencer**, dans le panneau de droite (Settings) : **Accelerator : GPU T4 x2** et **Internet : On**.

Kaggle ne voit pas ton Google Drive : les résultats sont écrits dans `/kaggle/working/results` et se téléchargent à la fin. Les runs Qwen ne font ici que l'expérience `promptfmt` (le reste est déjà fait).

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/patchbench && git clone -q https://github.com/Yonet28/HW2_NLP /kaggle/working/patchbench
%cd /kaggle/working/patchbench
!grep -c promptfmt_jobs patchbench/experiments.py   # doit afficher 2, sinon le repo n'est pas à jour
!pip install -q -r requirements.txt

GPU 0 : Llama 3B (environ 1 h 30). GPU 1 : `promptfmt` pour Qwen 3B puis Qwen 1.5B (environ 1 h 15). Les deux tournent en arrière-plan.

In [ ]:
import os
os.makedirs('/kaggle/working/results', exist_ok=True)
!CUDA_VISIBLE_DEVICES=0 nohup python -m patchbench.run --model unsloth/Llama-3.2-3B-Instruct --preset lean --experiments main,oracle --batch_size 2 --out /kaggle/working/results > /kaggle/working/log_llama.txt 2>&1 &
!CUDA_VISIBLE_DEVICES=1 nohup sh -c "python -m patchbench.run --model Qwen/Qwen2.5-3B-Instruct --preset standard --experiments promptfmt --batch_size 2 --out /kaggle/working/results && python -m patchbench.run --model Qwen/Qwen2.5-1.5B-Instruct --preset standard --experiments promptfmt --batch_size 4 --out /kaggle/working/results" > /kaggle/working/log_promptfmt.txt 2>&1 &

Relance cette cellule de temps en temps pour suivre la progression.

In [ ]:
!tail -n 3 /kaggle/working/log_llama.txt; echo; tail -n 3 /kaggle/working/log_promptfmt.txt
!wc -l /kaggle/working/results/*.jsonl

## Télécharger les résultats

In [ ]:
!cd /kaggle/working && zip -qr results.zip results log_*.txt && ls -la results.zip
from IPython.display import FileLink
FileLink('results.zip')